In [1]:
# Set project paths.
from pathlib import Path
import os
import sys

def find_project_root():
    current = Path.cwd()

    for folder in [current] + list(current.parents):
        if (folder / "Data").exists() and (folder / "Notebooks").exists():
            return folder

    raise FileNotFoundError("Could not find project root. Make sure Data and Notebooks folders exist.")

project_folder = find_project_root()
notebook_folder = project_folder / "Notebooks"

os.chdir(project_folder)

print("Project folder:", project_folder)
print("Notebook folder:", notebook_folder)

Project folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project
Notebook folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project/Notebooks


In [2]:
# Import packages.
import pandas as pd
import numpy as np
import holidays

In [3]:
# Define calendar features.
class HolidayFeatures:

    @staticmethod
    def get_uk_holidays():
        uk_holidays = holidays.CountryHoliday(
            "UK",
            subdiv="England",
            years=[2021, 2022, 2023],
        )

        holiday_dates = list(uk_holidays.keys())

        extra_bank_holidays = [
            pd.to_datetime("2022-06-02").date(),
            pd.to_datetime("2022-06-03").date(),
            pd.to_datetime("2022-09-19").date(),
            pd.to_datetime("2023-05-08").date(),
        ]

        holiday_dates.extend(extra_bank_holidays)

        closure_periods = pd.concat([
            pd.Series(pd.date_range(start="2022-12-23", end="2023-01-03")),
            pd.Series(pd.date_range(start="2023-12-23", end="2024-01-02")),
        ]).dt.date.tolist()

        holiday_dates.extend(closure_periods)

        return set(holiday_dates)

    @staticmethod
    def is_term_time(date_obj):
        m = date_obj.month
        d = date_obj.day

        if (m == 6 and d > 15) or m == 7 or m == 8 or (m == 9 and d < 25):
            return 0

        if (m == 3 and d > 25) or (m == 4 and d < 20):
            return 0

        if (m == 12 and d > 15) or (m == 1 and d < 15):
            return 0

        return 1

In [4]:
# Define consumption features.
class ConFeatures:

    @classmethod
    def create_consumption_features(cls, deop, solcast, steps_per_day=288, days_ahead=1):
        features = pd.DataFrame(index=deop.index)

        features["air_temp"] = solcast["air_temp"]
        features["heating_demand"] = np.maximum(0, 20 - features["air_temp"])
        features["cooling_demand"] = np.maximum(0, features["air_temp"] - 20)

        features["hour"] = features.index.hour
        features["day_of_week"] = features.index.dayofweek
        features["month"] = features.index.month
        features["is_weekend"] = features.index.dayofweek.isin([5, 6]).astype(int)

        holiday_set = HolidayFeatures.get_uk_holidays()

        features["is_holiday"] = features.index.date
        features["is_holiday"] = features["is_holiday"].apply(
            lambda d: 1 if d in holiday_set else 0
        )

        features["non_working_day"] = np.maximum(
            features["is_weekend"],
            features["is_holiday"],
        )

        features["non_working_hour_interaction"] = (
            features["non_working_day"] * features["hour"]
        )

        features["is_term_time"] = features.index.date
        features["is_term_time"] = features["is_term_time"].apply(
            HolidayFeatures.is_term_time
        )

        time_float = features.index.hour + features.index.minute / 60.0

        features["hour_sin"] = np.sin(2 * np.pi * time_float / 24)
        features["hour_cos"] = np.cos(2 * np.pi * time_float / 24)

        features["day_sin"] = np.sin(2 * np.pi * features.index.dayofweek / 7)
        features["day_cos"] = np.cos(2 * np.pi * features.index.dayofweek / 7)

        base_shift = steps_per_day * days_ahead

        features["load_lag_1d"] = deop["power-con-ave"].shift(base_shift)

        shift_7d = max(7, days_ahead) * steps_per_day
        features["load_lag_7d"] = deop["power-con-ave"].shift(shift_7d)

        yesterday_non_working = features["non_working_day"].shift(base_shift).fillna(0)

        features["smart_lag"] = np.where(
            (features["non_working_day"] == 0) & (yesterday_non_working == 0),
            features["load_lag_1d"],
            features["load_lag_7d"],
        )

        daily_mean = deop["power-con-ave"].resample("D").mean()
        daily_max = deop["power-con-ave"].resample("D").max()

        features["yesterday_daily_mean"] = features.index.normalize().map(
            daily_mean.shift(days_ahead)
        )

        features["yesterday_daily_max"] = features.index.normalize().map(
            daily_max.shift(days_ahead)
        )

        rolling_7d_mean = daily_mean.rolling(7).mean()

        features["rolling_7d_baseline"] = features.index.normalize().map(
            rolling_7d_mean.shift(days_ahead)
        )

        return features.dropna()

In [5]:
# Define solar features.
class PVFeatures:

    @classmethod
    def create_pv_features(cls, deop, solcast, days_ahead=1, steps_per_day=288):
        df = pd.DataFrame(index=deop.index)

        df["gti_target"] = solcast["gti"]
        df["cloud_opacity_target"] = solcast["cloud_opacity"]
        df["snow_depth"] = solcast["snow_depth"]

        base_shift = steps_per_day * days_ahead

        df["month"] = deop.index.month
        df["day_of_month"] = deop.index.day

        df["lag_1d"] = deop["power-gen-pv-ave"].shift(base_shift)
        df["lag_2d"] = deop["power-gen-pv-ave"].shift(base_shift + steps_per_day)
        df["lag_3d"] = deop["power-gen-pv-ave"].shift(base_shift + steps_per_day * 2)

        shift_7d = max(7, days_ahead) * steps_per_day
        shift_14d = max(14, days_ahead) * steps_per_day

        df["lag_7d"] = deop["power-gen-pv-ave"].shift(shift_7d)
        df["lag_14d"] = deop["power-gen-pv-ave"].shift(shift_14d)

        df["power_volatility_1d"] = (
            deop["power-gen-pv-ave"].shift(base_shift).rolling(24).std()
        )

        df["power_trend_1d"] = df["lag_1d"] - df["lag_2d"]

        df["gti_volatility_1d"] = (
            solcast["gti"].shift(base_shift).rolling(24).std()
        )

        daily_max = deop["power-gen-pv-ave"].resample("D").max().shift(days_ahead)

        df["yesterday_peak"] = df.index.normalize().map(daily_max)

        hour_num = df.index.hour + df.index.minute / 60

        df["solar_potential"] = np.maximum(
            0,
            np.sin(np.pi * (hour_num - 6) / 12),
        )

        df["hour_sin"] = np.sin(2 * np.pi * df.index.hour / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df.index.hour / 24)

        return df.dropna()

In [6]:
# Test feature tables.
deop = pd.read_csv(
    "Data/DEOP/2023_DEOP_Repaired.csv",
    parse_dates=["DateTime"],
    index_col="DateTime",
)

solcast = pd.read_csv(
    "Data/Solcast/Solcast_2023.csv",
    parse_dates=["DateTime"],
    index_col="DateTime",
)

con_test = ConFeatures.create_consumption_features(deop, solcast)
pv_test = PVFeatures.create_pv_features(deop, solcast)

print(con_test.head())
print(pv_test.head())

print("Consumption feature shape:", con_test.shape)
print("PV feature shape:", pv_test.shape)

/var/folders/yd/5b21hsj56kn7zqtz63fs0chh0000gn/T/ipykernel_4129/1837063544.py:6: DeprecationWarning: CountryHoliday is deprecated, use country_holidays instead.
  uk_holidays = holidays.CountryHoliday(


                     air_temp  heating_demand  cooling_demand  hour  \
DateTime                                                              
2023-01-08 00:05:00         7              13               0     0   
2023-01-08 00:10:00         7              13               0     0   
2023-01-08 00:15:00         7              13               0     0   
2023-01-08 00:20:00         7              13               0     0   
2023-01-08 00:25:00         7              13               0     0   

                     day_of_week  month  is_weekend  is_holiday  \
DateTime                                                          
2023-01-08 00:05:00            6      1           1           0   
2023-01-08 00:10:00            6      1           1           0   
2023-01-08 00:15:00            6      1           1           0   
2023-01-08 00:20:00            6      1           1           0   
2023-01-08 00:25:00            6      1           1           0   

                     non_working